# Regularized Models Notebook

This notebook follows the proposal's **Week 4-5** milestones:
- evaluate multiple target audio features,
- compare `Dummy`, `Linear`, `Ridge`, and `Lasso`,
- tune regularization strengths,
- export comparison tables for final reporting.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 100)
np.random.seed(42)


In [ ]:
SRC_PATH = Path.cwd() / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from project_utils import compute_regression_metrics, ensure_dir, ensure_processed_data


In [ ]:
data_parts = ensure_processed_data(data_dir="data", processed_dir="data/processed", random_state=42)
train_df = data_parts["train"]
val_df = data_parts["val"]
test_df = data_parts["test"]

print("Train/Val/Test shapes:", train_df.shape, val_df.shape, test_df.shape)


In [ ]:
proposal_targets = [
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "loudness",
    "tempo",
]

available_targets = [t for t in proposal_targets if t in train_df.columns]
if not available_targets:
    raise ValueError("No proposal targets found in the processed dataset.")

available_targets


In [ ]:
def build_models():
    return {
        "mean_baseline": DummyRegressor(strategy="mean"),
        "linear_regression": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("linearregression", LinearRegression()),
            ]
        ),
        "ridge": Pipeline(
            [
                ("scaler", StandardScaler()),
                ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 30), cv=5)),
            ]
        ),
        "lasso": Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "lasso",
                    LassoCV(
                        alphas=np.logspace(-4, 1, 40),
                        cv=5,
                        max_iter=20000,
                        random_state=42,
                    ),
                ),
            ]
        ),
    }

rows = []
param_rows = []

for target in available_targets:
    feature_cols = [c for c in train_df.columns if c != target]

    X_train, y_train = train_df[feature_cols], train_df[target]
    X_val, y_val = val_df[feature_cols], val_df[target]
    X_test, y_test = test_df[feature_cols], test_df[target]

    for model_name, model in build_models().items():
        fitted = model.fit(X_train, y_train)

        for split_name, X_split, y_split in [
            ("validation", X_val, y_val),
            ("test", X_test, y_test),
        ]:
            y_pred = fitted.predict(X_split)
            metrics = compute_regression_metrics(y_split, y_pred)
            rows.append(
                {
                    "target": target,
                    "model": model_name,
                    "split": split_name,
                    **metrics,
                    "n_features": len(feature_cols),
                }
            )

        alpha = None
        if model_name == "ridge":
            alpha = fitted.named_steps["ridge"].alpha_
        elif model_name == "lasso":
            alpha = fitted.named_steps["lasso"].alpha_

        if alpha is not None:
            param_rows.append({"target": target, "model": model_name, "best_alpha": float(alpha)})

results_df = pd.DataFrame(rows)
params_df = pd.DataFrame(param_rows)

results_df.head()


In [ ]:
outputs_dir = ensure_dir("outputs")
results_path = outputs_dir / "regularized_results.csv"
params_path = outputs_dir / "regularized_best_params.csv"
test_results_path = outputs_dir / "regularized_test_results.csv"

results_df.to_csv(results_path, index=False)
params_df.to_csv(params_path, index=False)
results_df[results_df["split"] == "test"].to_csv(test_results_path, index=False)

print(f"Saved -> {results_path}")
print(f"Saved -> {params_path}")
print(f"Saved -> {test_results_path}")


In [ ]:
test_results = results_df[results_df["split"] == "test"].copy()
r2_pivot = test_results.pivot(index="target", columns="model", values="r2")
r2_pivot


In [ ]:
ax = r2_pivot.plot(kind="bar", figsize=(12, 6))
ax.set_ylabel("R² (higher is better)")
ax.set_title("Test R² by Target and Model")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
best_by_target = (
    test_results.sort_values("r2", ascending=False)
    .groupby("target", as_index=False)
    .first()
    .sort_values("r2", ascending=False)
)

best_path = outputs_dir / "best_model_by_target.csv"
best_by_target.to_csv(best_path, index=False)
print(f"Saved -> {best_path}")

best_by_target


## Next Notebook
Run [final.ipynb](final.ipynb) to produce the final comparison charts and submission summary.
